# REPRODUCE_ALL — Phase 1 + 1.5 full re-run with committed JSON artifacts

Upload the whole `sideproject` folder to Google Drive, open this notebook in Colab,
pick a GPU runtime, and **Run All** (or run section-by-section).

**What it does.** Re-trains every experiment from scratch and writes a JSON metric file
for each, so every documented number becomes independently verifiable. The audit
(2026-06-03) found 9/10 notebooks were unexecuted skeletons with zero committed metrics.
The final **Section 10** prints a documented-vs-measured table and saves `out/VERIFICATION.json`.

**Sections are independent** (each wrapped in try/except) — one failure won't block the rest.
Paste any traceback back and we'll fix it. Heaviest cost = `ph1_v6_long` (500 epochs) and the
1a robustness grid; toggle them off in the config cell if you want a faster first pass.

**Corpus note.** Phase-1 recon defaults to the *current* code corpus (`default4`). The recorded
v3–v6 numbers came from a pre-2026-05-26 *7-source* corpus, so recon/SimBench deltas for those
runs are EXPECTED (the corpus default flipped) — that is itself a finding, surfaced in Section 10.

In [ ]:
# ============================== MASTER CONFIG ==============================
BASE = '/content/drive/MyDrive/sideproject'   # <-- change if your Drive path differs

# --- section toggles ---
RUN_PHASE1_RECON   = True
RUN_BASELINES      = True
RUN_SIMBENCH       = True
RUN_ENGINE_A       = True
RUN_1A             = True
RUN_1A_ROBUSTNESS  = True
RUN_1B             = True
RUN_INTERVENTION   = True

# --- sub-flags for the expensive bits ---
RUN_V6_LONG          = True    # ph1_v6_long = 500 epochs (biggest single cost)
RUN_ENGINE_A_LEVERS  = True    # E3 F3 levers + E4 encoder-swap ceiling

# --- confound-fix cells (Section 9) ---
INCLUDE_CONFOUND_FIXES  = True
FIX_SIMBENCH_MAJORITY   = True   # add random + majority base-rate baseline
FIX_1A_SAME_AXIS        = True   # operation_gate adj vs operation_ceiling_raw adj, same axis
FIX_1B_CHAINSTEPS_4     = True   # re-run 1b at chain_steps=4 (matches 4-hop) + multi-seed
FIX_1B_MOTIF_TOPIC_CTRL = True   # report motif op_purity vs op_chance

# --- reproducibility ---
PHASE1_RECON_CORPUS = 'default4'  # 'default4' (current code, robust) | 'legacy7' (historical 7-source; eval match approximate)
B1_CONTINUE_ON_SANITY_FAIL = False  # False = let B1 sanity-fail on expert collapse (the documented result; no model).
                                    #   True = force B1 through training despite collapse (gives a model for SimBench compare).
REUSE_EXISTING = True             # if out/phase1/<run_id>/model.pt already exists on Drive, SKIP training and just eval.
                                  #   -> reuses the checkpoints your earlier Colab runs auto-saved (v6 won't retrain 500 ep).
FRESH       = False               # only used when actually training: True wipes the run dir first (defeats ckpt resume).
RUN_SUFFIX  = ''                  # '' = use the ORIGINAL run_ids (ph1_v6_long, ...) so existing Drive checkpoints are found.
                                  #   set to e.g. '_repro' to force brand-new dirs (ignores your existing checkpoints).
SEEDS       = [0, 1, 2]           # 1a robustness + 1b chain_steps=4 multi-seed
print('config loaded.')

## Section 0 — Preflight (mount, GPU caps, smoke tests, harness)

In [ ]:
# 0.0 mount + chdir + sys.path  (keep cwd = BASE so out/phase1 and out/phase1_5 share one tree)
import os, sys, shutil, json, traceback
from pathlib import Path
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except Exception as e:
        print('drive mount skipped (not on Colab?):', e)
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

In [ ]:
# 0.1 version print (no auto-pip; if an import fails below, install manually then re-run)
import importlib
for m in ['torch', 'transformers', 'datasets', 'numpy', 'sklearn', 'pandas']:
    try:
        mod = importlib.import_module(m); print(f'{m:14s} {getattr(mod, "__version__", "?")}')
    except Exception as e:
        print(f'{m:14s} MISSING  ->  pip install {m}   ({e})')

In [ ]:
# 0.2 GPU detect -> batch/sample caps
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
# V6_EPOCHS=80: the recorded v6_long run was stopped at epoch 79, and recon plateaus by
# ~epoch 15, so 80 captures the documented number in ~1h instead of 5h at 500.
if 'A100' in gpu:
    TIER='A100'; BATCH_PHASE1=64; BATCH_PHASE15=128; ENGINE_A_BS=64; SAMPLE_CAP_15=20000; V6_EPOCHS=80; ENGINE_A_EPOCHS=40
elif any(x in gpu for x in ['L4','A10','V100','RTX','L40']):
    TIER='L4';  BATCH_PHASE1=32; BATCH_PHASE15=64;  ENGINE_A_BS=64; SAMPLE_CAP_15=20000; V6_EPOCHS=80; ENGINE_A_EPOCHS=40
elif 'T4' in gpu:
    TIER='T4';  BATCH_PHASE1=32; BATCH_PHASE15=32;  ENGINE_A_BS=32; SAMPLE_CAP_15=8000;  V6_EPOCHS=80; ENGINE_A_EPOCHS=25
else:
    TIER='CPU'; BATCH_PHASE1=16; BATCH_PHASE15=16;  ENGINE_A_BS=16; SAMPLE_CAP_15=2000;  V6_EPOCHS=20; ENGINE_A_EPOCHS=10
if not RUN_V6_LONG:
    V6_EPOCHS = 30
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
PH1 = Path('out/phase1'); P15 = Path('out/phase1_5'); PHASE15_CACHE = P15 / 'cache'
PH1.mkdir(parents=True, exist_ok=True); PHASE15_CACHE.mkdir(parents=True, exist_ok=True)
print(f'GPU={gpu} | TIER={TIER} | DEV={DEV}')
print(f'BATCH_PHASE1={BATCH_PHASE1} BATCH_PHASE15={BATCH_PHASE15} ENGINE_A_BS={ENGINE_A_BS} '
      f'SAMPLE_CAP_15={SAMPLE_CAP_15} V6_EPOCHS={V6_EPOCHS} ENGINE_A_EPOCHS={ENGINE_A_EPOCHS}')
if TIER == 'CPU':
    print('WARNING: no GPU detected — training will be very slow; this is smoke-only.')

In [ ]:
# 0.3 HF dataset reachability smoke (no auth needed). Failures are warnings, not fatal.
def _smoke():
    import pandas as pd
    from datasets import load_dataset
    checks = [
        ('MuSiQue', lambda: load_dataset('dgslibisey/MuSiQue', split='train', streaming=True)),
        ('Super-NI', lambda: load_dataset('Muennighoff/natural-instructions', split='train',
                                          streaming=True, verification_mode='no_checks')),
        ('QuAIL',   lambda: load_dataset('textmachinelab/quail', split='train', streaming=True)),
    ]
    for name, fn in checks:
        try:
            next(iter(fn().take(1))); print(f'  {name:10s} OK')
        except Exception as e:
            print(f'  {name:10s} UNREACHABLE: {type(e).__name__}: {e}')
    try:
        url = 'https://huggingface.co/datasets/pitehu/SimBench/resolve/main/SimBenchPop.csv'
        pd.read_csv(url, nrows=3); print('  SimBench   OK')
    except Exception as e:
        print(f'  SimBench   UNREACHABLE: {type(e).__name__}: {e}')
_smoke()

In [ ]:
# 0.4 import smoke for every module the notebook touches (catches a stale Drive sync early)
mods = [
    'experiments.phase1.train', 'experiments.phase1.eval', 'experiments.phase1.eval_simbench_classifier', 'experiments.phase1.cluster_analysis',
    'experiments.phase1.engine_a', 'experiments.phase1.eval_opcycle', 'experiments.phase1.data', 'experiments.phase1.baselines.train_baselines',
    'experiments.phase1.baselines.generic_baseline',
    'experiments.phase1_5.engine_1a', 'experiments.phase1_5.ablations', 'experiments.phase1_5.run_1b', 'experiments.phase1_5.eval',
    'experiments.phase1_5.data', 'experiments.phase1_5.intervention', 'experiments.phase1_5.model', 'experiments.phase1_5.train',
]
ok = True
for m in mods:
    try:
        importlib.import_module(m)
    except Exception as e:
        ok = False; print(f'IMPORT FAIL {m}: {type(e).__name__}: {e}')
print('all imports OK' if ok else 'FIX IMPORTS ABOVE before running sections')

In [ ]:
# 0.5 + 0.6 helpers: fresh() wipe + run_section() harness
RESULTS = {}

def fresh(run_dir):
    p = Path(run_dir)
    if FRESH and p.exists():
        shutil.rmtree(p)

def run_section(name, fn):
    print('\n' + '=' * 72 + f'\n[{name}] starting\n' + '=' * 72)
    try:
        fn(); RESULTS[name] = 'OK'; print(f'[{name}] DONE')
    except Exception as e:
        RESULTS[name] = f'FAIL: {type(e).__name__}: {e}'
        print(f'[{name}] FAILED — full traceback:'); traceback.print_exc()
    finally:
        # Free GPU between sections so a trained model from one section doesn't
        # starve the next (the chain_steps=4 run is memory-heavy).
        try:
            import gc, torch as _t
            gc.collect()
            if _t.cuda.is_available():
                _t.cuda.empty_cache()
        except Exception:
            pass

def rid(base):
    return base + RUN_SUFFIX

LEGACY7_CACHE = str(PH1 / 'cache' / 'corpus_legacy7.parquet')

# Degenerate-label guard for experiments.phase1.eval.linear_probe_separability.
# eval.py:174 hardcodes a `source=="pennebaker"` probe label, but pennebaker is
# absent from the current default (4-source) corpus -> single-class label ->
# LinearSVC crashes and takes the (already-computed) recon number down with it.
# The source-probe is a secondary sanity metric; skip it on <2 classes so recon
# survives. (No repo edit needed — patches the module attribute eval_phase1 looks up.)
import numpy as _np
import experiments.phase1.eval as _ph1eval
if not getattr(_ph1eval, '_lps_patched', False):
    _orig_lps = _ph1eval.linear_probe_separability
    def _safe_lps(features, labels, *a, **kw):
        if len(set(_np.asarray(labels).tolist())) < 2:
            return {'skipped': 'single-class probe label (absent in current corpus)',
                    'n_classes': 1, 'accuracy': None, 'macro_f1': None,
                    'n_samples': int(_np.asarray(labels).shape[0])}
        return _orig_lps(features, labels, *a, **kw)
    _ph1eval.linear_probe_separability = _safe_lps
    _ph1eval._lps_patched = True
print('helpers ready (linear-probe degenerate-label guard active).')

## Section 1 — Phase 1 recon-cycle (v3 / v4 / v5 / v6)
Trains each variant, saves `eval.json` (recon cosine + linear probes) and `cluster.json`
(expert clustering — `analyze_experts` otherwise only prints).

In [ ]:
def _phase1_recon():
    from experiments.phase1.train import train_phase1
    from experiments.phase1.eval import eval_phase1
    from experiments.phase1.cluster_analysis import analyze_experts
    from experiments.phase1.data import build_corpus, legacy_v6_corpus_config

    corpus_cache = None
    if PHASE1_RECON_CORPUS == 'legacy7':
        if not Path(LEGACY7_CACHE).exists():
            print('[recon] building legacy 7-source corpus parquet (one-time)...')
            build_corpus(legacy_v6_corpus_config(), cache_path=LEGACY7_CACHE)
        corpus_cache = LEGACY7_CACHE
        print('[recon] NOTE legacy7: eval_phase1 rebuilds the default corpus internally, so its '
              'recon split may not exactly match the legacy7 training split (approximate).')

    runs = [
        ('ph1_v3_minimal', dict(epochs=30,        lambda_lb=0.10, lambda_ortho=0.00)),
        ('ph1_v4_diverse', dict(epochs=30,        lambda_lb=0.10, lambda_ortho=0.00)),  # == v3 in committed code
        ('ph1_v5_arch',    dict(epochs=30,        lambda_lb=0.05, lambda_ortho=0.05)),
    ]
    if RUN_V6_LONG:
        runs.append(('ph1_v6_long', dict(epochs=V6_EPOCHS, lambda_lb=0.10, lambda_ortho=0.00)))

    for base, kw in runs:
        r = rid(base)
        if REUSE_EXISTING and (PH1 / r / 'model.pt').exists():
            print(f'\n--- {r}: REUSE existing checkpoint on Drive (skip training) ---')
        else:
            if FRESH:
                fresh(PH1 / r)
            print(f'\n--- training {r} ({kw}) ---')
            train_phase1(run_id=r, batch_size=BATCH_PHASE1, lr=1e-4, warmup_steps=300,
                         grad_clip=1.0, corpus_cache=corpus_cache, **kw)
        ev = eval_phase1(run_id=r)
        cl = analyze_experts(r)
        (PH1 / r / 'cluster.json').write_text(json.dumps(cl, indent=2, default=str))
        print(f'[{r}] recon_cos={ev["cycle_reconstruction"]["mean_cosine"]:.4f} '
              f'alpha_f1={ev["linear_probe_routed_alpha"].get("macro_f1")} '
              f'subkg_f1={ev["linear_probe_sub_kg"].get("macro_f1")}')

if RUN_PHASE1_RECON:
    run_section('S1 phase1-recon', _phase1_recon)

## Section 2 — Baselines B0 / B1  (closes "B1 was never run")
B1 ("Switch top-1 expert collapse") was reported as a predicted failure but never executed.
Here it runs, and the K_active collapse is *measured* from history.json, not asserted.

In [ ]:
def _baselines():
    import torch, torch.nn.functional as F
    from torch.utils.data import DataLoader
    from experiments.phase1.baselines.train_baselines import train_baseline
    from experiments.phase1.baselines.generic_baseline import GenericBaseline
    from experiments.phase1.data import CorpusConfig, Phase1Dataset, load_phase1_corpus
    from experiments.phase1.train import CONFIG_NAME, MODEL_NAME

    b0, b1 = rid('b0_v0'), rid('b1_v0')
    if REUSE_EXISTING and (PH1 / b0 / 'model.pt').exists():
        print('--- B0: REUSE existing checkpoint (skip training) ---')
    else:
        if FRESH:
            fresh(PH1 / b0)
        print('--- B0 (training) ---')
        train_baseline(baseline='B0', run_id=b0, epochs=30, batch_size=BATCH_PHASE1, lr=1e-4,
                       warmup_steps=300, grad_clip=1.0, mlp_width=4500, mlp_n_hidden=3)
    if REUSE_EXISTING and (PH1 / b1 / 'model.pt').exists():
        print('--- B1: REUSE existing checkpoint (skip training) ---')
    else:
        if FRESH:
            fresh(PH1 / b1)
        print('--- B1 (training — the never-run baseline; may sanity-fail on collapse) ---')
        try:
            train_baseline(baseline='B1', run_id=b1, epochs=30, batch_size=BATCH_PHASE1, lr=1e-4,
                           warmup_steps=300, grad_clip=1.0, k_routed=20, d_hidden=2048, lambda_lb=0.1,
                           continue_on_sanity_fail=B1_CONTINUE_ON_SANITY_FAIL)
        except Exception as e:
            print(f'[B1] train_baseline aborted: {type(e).__name__}: {e}')

    # B0 cycle-recon eval (06_baselines.ipynb cell-3 pattern) -> eval.json
    device = torch.device(DEV)
    run_dir = PH1 / b0
    cfg = json.loads((run_dir / CONFIG_NAME).read_text())
    ccfg = CorpusConfig(encoder_name=cfg['encoder_name'], encoder_max_length=cfg['encoder_max_length'])
    corpus, fact_emb = load_phase1_corpus(cfg=ccfg)
    model = GenericBaseline(encoder_name=cfg['encoder_name'], d_bottleneck=cfg.get('d_bottleneck', 64),
                            mlp_width=cfg.get('mlp_width', 4500), mlp_n_hidden=cfg.get('mlp_n_hidden', 3)).to(device)
    model.load_state_dict(torch.load(run_dir / MODEL_NAME, map_location=device), strict=False)
    model.eval()
    tmask = (corpus['split'] == 'test').values
    loader = DataLoader(Phase1Dataset(fact_emb[tmask], corpus['user_id'].values[tmask]),
                        batch_size=128, shuffle=False)
    tc = tm = 0.0; n = 0
    with torch.no_grad():
        for fact, _ in loader:
            fact = fact.to(device); recon = model(fact)['recon']
            tc += F.cosine_similarity(fact, recon, dim=-1).sum().item()
            tm += ((fact - recon) ** 2).mean(dim=-1).sum().item(); n += fact.size(0)
    payload = {'cycle_reconstruction': {'mean_cosine': tc / n, 'mean_mse': tm / n}}
    ej = run_dir / 'eval.json'
    if ej.exists():
        ex = json.loads(ej.read_text()); ex.update(payload); payload = ex
    ej.write_text(json.dumps(payload, indent=2))
    print(f'[{b0}] recon_cos={tc / n:.4f}')

    # B1 expert-collapse MEASUREMENT — history.json if it trained through, else
    # sanity.json if it aborted on the (documented) collapse.
    hp, sp = PH1 / b1 / 'history.json', PH1 / b1 / 'sanity.json'
    if hp.exists():
        hh = json.loads(hp.read_text())
        print(f'[{b1}] K_active(last) = {hh[-1].get("active_frac", 0.0) * 20:.2f} of 20  '
              f'(collapse if ~1.0)  [trained through]')
    elif sp.exists():
        ss = json.loads(sp.read_text())
        print(f'[{b1}] SANITY-FAILED (training aborted): verdict={ss.get("verdict")} '
              f'k_active_mean={ss.get("k_active_mean")} reasons={ss.get("reasons")}')
        print('       -> documented "B1 Switch top-1 expert collapse" now MEASURED, not predicted')
    else:
        print(f'[{b1}] no history/sanity — B1 did not run')

if RUN_BASELINES:
    run_section('S2 baselines', _baselines)

## Section 3 — SimBench downstream classifier (all 6 runs)

In [ ]:
def _simbench():
    from experiments.phase1.eval_simbench_classifier import train_simbench_classifier
    bases = ['ph1_v3_minimal', 'ph1_v4_diverse', 'ph1_v5_arch']
    if RUN_V6_LONG:
        bases.append('ph1_v6_long')
    bases += ['b0_v0', 'b1_v0']
    for base in bases:
        r = rid(base)
        if not (PH1 / r / 'model.pt').exists():
            print(f'[simbench {r}] SKIP (no model.pt — run its training section first)'); continue
        try:
            res = train_simbench_classifier(run_id=r, epochs=100, batch_size=256, lr=1e-3,
                                            hidden=512, dropout=0.1)
            print(f'[{r}] argmax_acc={res["test"]["argmax_acc"]:.4f} kl={res["test"]["kl"]:.4f}')
        except Exception as e:
            print(f'[simbench {r}] FAIL {type(e).__name__}: {e}')

if RUN_SIMBENCH:
    run_section('S3 simbench', _simbench)

## Section 4 — Engine-A operation cycle (pivot 1: F3 diagnosis)
Super-NI training corpus → QuAIL probe → raw-encoder **ceiling** (E1: ctx-first vs q-first) →
clean Stage-1 run (question-first, T=256). Saves `ceiling.json`, `report_stage1.json` (with
the matched `ceiling_0b_adj` beside the executed `adj_operation`), and optionally F3 levers +
encoder-swap ceiling.

In [ ]:
def _engine_a():
    import numpy as np, pandas as pd, torch
    from sklearn.cluster import KMeans
    from experiments.phase1.data import CorpusConfig, build_corpus, encode_or_load_tokens, load_quail_probe
    from experiments.phase1.engine_a import run_engine_a
    from experiments.phase1.eval_opcycle import raw_embedding_report

    ccfg = CorpusConfig(include_news=False, include_arxiv=False, include_tweet=False,
                        include_amazon=False, include_reddit=False, include_superni=True,
                        superni_max_samples=40_000, superni_max_per_task=50)
    corpus = build_corpus(ccfg)
    train_tokens, train_mask = encode_or_load_tokens(corpus, t_cap=128, batch_size=ENGINE_A_BS)
    print('train tokens', train_tokens.shape)

    def build_probe(question_first, t_cap, encoder_name='BAAI/bge-large-en-v1.5'):
        p = load_quail_probe(ccfg, question_first=question_first).head(ccfg.quail_max_samples).reset_index(drop=True)
        ptok, pmask = encode_or_load_tokens(p, encoder_name=encoder_name, t_cap=t_cap, batch_size=ENGINE_A_BS)
        op, op_u = pd.factorize(p['label'])
        pooled = ((ptok.astype('float32') * pmask[..., None]).sum(1) / pmask.sum(1, keepdims=True).clip(min=1))
        topic = KMeans(n_clusters=min(8, len(op_u) + 2), n_init=10, random_state=0).fit_predict(pooled)
        edges = np.unique(np.quantile(pmask.sum(1), [.25, .5, .75])); tt = np.digitize(pmask.sum(1), edges)
        ctrls = {'topic': topic}
        if np.unique(tt).size >= 2:
            ctrls['token_type'] = tt
        return p, ptok, pmask, op, op_u, ctrls

    out = PH1 / 'engine_a'; out.mkdir(parents=True, exist_ok=True)

    def ceiling(qf, tcap, tag):
        _, ptok, pmask, op, op_u, ctrls = build_probe(qf, tcap)
        rep = raw_embedding_report(ptok, pmask, op, control_label_sets=ctrls, agg='meanmax', seed=0)
        print(f'[{tag}] adj_op={rep["adj_operation"]:.4f} topic_sel={rep["selectivity"].get("topic", float("nan")):+.4f}')
        return rep
    rep_0a = ceiling(False, 128, '0a ctx-first T128')
    rep_0b = ceiling(True, 256, '0b q-first  T256')
    keep = ('acc_operation', 'adj_operation', 'selectivity')
    (out / 'ceiling.json').write_text(json.dumps(
        {'rep_0a': {k: rep_0a[k] for k in keep}, 'rep_0b': {k: rep_0b[k] for k in keep}},
        indent=2, default=str))

    probe, ptok, pmask, op_labels, op_uniques, controls = build_probe(True, 256)
    result = run_engine_a(train_tokens, train_mask, ptok, pmask, op_labels,
                          control_label_sets=controls, d_z=256, k=16, d_hidden=512,
                          epochs=ENGINE_A_EPOCHS, batch_size=ENGINE_A_BS * 2, lr=1e-3, seed=0,
                          agg='meanmax', k_target=4.0, log_every=5, device=DEV)
    rep = result['report']
    (out / 'report_stage1.json').write_text(json.dumps({
        'verdict': rep['verdict'], 'acc_operation': rep['acc_operation'],
        'adj_operation': rep['adj_operation'], 'controls': rep['controls'],
        'selectivity': rep['selectivity'], 'warnings': rep['warnings'],
        'operations': list(map(str, op_uniques)), 'history': result['history'],
        'probe': 'question_first T256', 'ceiling_0b_adj': rep_0b['adj_operation'],
    }, indent=2, default=str))
    print('VERDICT', rep['verdict'], '| adj_op', round(rep['adj_operation'], 4),
          '| ceiling_0b_adj', round(rep_0b['adj_operation'], 4))

    if RUN_ENGINE_A_LEVERS:
        levers = []
        for tag, kw in [('best_recon', dict(use_best_recon=True)),
                        ('deviation',  dict(route_on_deviation=True)),
                        ('both',       dict(use_best_recon=True, route_on_deviation=True))]:
            r = run_engine_a(train_tokens, train_mask, ptok, pmask, op_labels,
                             control_label_sets=controls, d_z=256, k=16, d_hidden=512,
                             epochs=ENGINE_A_EPOCHS, batch_size=ENGINE_A_BS * 2, lr=1e-3, seed=0,
                             agg='meanmax', k_target=4.0, log_every=0, device=DEV, **kw)
            rr = r['report']
            levers.append({'lever': tag, 'verdict': rr['verdict'], 'adj_operation': rr['adj_operation'],
                           'topic_sel': rr['selectivity'].get('topic')})
            print(f'[F3:{tag:10s}] {rr["verdict"]} adj={rr["adj_operation"]:.4f}')
        (out / 'f3_levers.json').write_text(json.dumps(levers, indent=2, default=str))
        try:
            _, pt2, pm2, op2, _, ctrls2 = build_probe(True, 256, encoder_name='intfloat/e5-large-v2')
            rep2 = raw_embedding_report(pt2, pm2, op2, control_label_sets=ctrls2, agg='meanmax', seed=0)
            (out / 'encoder_swap_ceiling.json').write_text(json.dumps(
                {'encoder': 'intfloat/e5-large-v2', 'adj_operation': rep2['adj_operation'],
                 'selectivity': rep2['selectivity']}, indent=2, default=str))
            print(f'[F2 e5 ceiling] adj={rep2["adj_operation"]:.4f}')
        except Exception as e:
            print('E4 encoder-swap ceiling skipped:', e)

if RUN_ENGINE_A:
    run_section('S4 engine-a', _engine_a)

## Section 5 — Phase 1.5 1a ablations (pivot 2)
`run_all_rows` auto-saves `row_A.{1,2,3}_*.json` + probe codes + `summary_seed0.csv`.

In [ ]:
def _ablations_1a():
    from experiments.phase1_5.data import MCCorpusConfig
    from experiments.phase1_5.ablations import PHASE1_5_INITIAL_ROWS, run_all_rows
    from experiments.phase1_5.train import TrainConfig
    cfg = MCCorpusConfig(max_train_samples=SAMPLE_CAP_15, max_val_samples=2000,
                         max_test_samples=2000, cache_root=str(PHASE15_CACHE))
    tc = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, seed=0)
    df = run_all_rows(PHASE1_5_INITIAL_ROWS, corpus_cfg=cfg, train_cfg=tc,
                      out_dir=str(P15 / 'ablations'), device=DEV, seed=0,
                      batch_size=BATCH_PHASE15, skip_if_exists=not FRESH)
    cols = [c for c in ['op_adj_operation', 'passes_sigma_gate', 'op_ceiling_raw_adj',
                        'adj_topic', 'val_mc_acc_final'] if c in df.columns]
    print(df[cols].to_string())

if RUN_1A:
    run_section('S5 1a-ablations', _ablations_1a)

## Section 6 — 1a robustness / top-k  (does the 1a negative survive seed variation?)
Replication grid: seed × use_best_val → `out/phase1_5/replication/`. Top-k arm → `out/phase1_5/topk/`.

In [ ]:
def _robustness():
    from experiments.phase1_5.data import MCCorpusConfig, MODE_Q_ONLY
    from experiments.phase1_5.ablations import AblationRow, run_ablation_row
    from experiments.phase1_5.model import MOD_KG_HYPERNET
    from experiments.phase1_5.train import TrainConfig
    cfg = MCCorpusConfig(max_train_samples=SAMPLE_CAP_15, max_val_samples=2000,
                         max_test_samples=2000, cache_root=str(PHASE15_CACHE))
    for seed in SEEDS:
        for ubv in (True, False):
            row = AblationRow(row_id=f'R_s{seed}_bv{int(ubv)}', name='replication',
                              encoding_mode=MODE_Q_ONLY, k_routed=64, lb_strategy='aux_free',
                              modulation=MOD_KG_HYPERNET)
            tc = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, use_best_val=ubv, seed=seed)
            run_ablation_row(row, corpus_cfg=cfg, train_cfg=tc, out_dir=str(P15 / 'replication'),
                             device=DEV, seed=seed, batch_size=BATCH_PHASE15, skip_if_exists=not FRESH)
    row = AblationRow(row_id='T_topk4', name='topk', encoding_mode=MODE_Q_ONLY, k_routed=64,
                      lb_strategy='aux_free', routing='topk', modulation=MOD_KG_HYPERNET)
    tc = TrainConfig(epochs=40, lr=1e-3, k_target=4.0, lam_z=1e-3, use_best_val=False, seed=0)
    run_ablation_row(row, corpus_cfg=cfg, train_cfg=tc, out_dir=str(P15 / 'topk'),
                     device=DEV, seed=0, batch_size=BATCH_PHASE15, skip_if_exists=not FRESH)

if RUN_1A_ROBUSTNESS:
    run_section('S6 1a-robustness', _robustness)

## Section 7 — Phase 1.5 1b MuSiQue chain-of-experts (pivot 3)
M3 flat / M5 chain(L=3) / M6 logic-MC control. `run_1b_experiment` returns but does NOT save,
so each result is explicitly written under `out/phase1_5/musique/`.

In [ ]:
def _musique_1b():
    from experiments.phase1_5.data import MCCorpusConfig
    from experiments.phase1_5.ablations import AblationRow, _json_default
    from experiments.phase1_5.engine_1a import run_engine_1a
    from experiments.phase1_5.run_1b import run_1b_experiment
    from experiments.phase1_5.train import TrainConfig
    mus = P15 / 'musique'; mus.mkdir(parents=True, exist_ok=True)
    cfg = MCCorpusConfig(corpus='musique', cache_root=str(PHASE15_CACHE),
                         max_train_samples=SAMPLE_CAP_15, max_val_samples=2000, max_test_samples=2000)

    print('--- M3 flat (chain_steps=1) ---')
    row = AblationRow(row_id='M3', name='MuSiQue flat-1a', k_routed=128, lb_strategy='aux_free',
                      corpus='musique', chain_steps=1, dropout=0.1)
    res1a = run_engine_1a(ablation_row=row, corpus_cfg=cfg,
                          train_cfg=TrainConfig(epochs=15, lr=1e-3, k_target=4.0, best_metric='acc', seed=0),
                          device=DEV, seed=0, batch_size=BATCH_PHASE15, out_dir=str(mus))
    (mus / 'M3_result.json').write_text(json.dumps(res1a, indent=2, default=_json_default))
    print(f'[M3] flat val_acc(last)={(res1a["history"] or [{}])[-1].get("val_mc_acc")}')

    print('--- M5 chain (chain_steps=3) ---')
    out1b = run_1b_experiment(corpus_cfg=cfg, train_cfg=TrainConfig(epochs=15, lr=1e-3, seed=0),
                              chain_steps=3, k_routed=128, k_active_target=4.0,
                              batch_size=BATCH_PHASE15, device=DEV, seed=0)
    (mus / 'M5_chain3_seed0.json').write_text(json.dumps(out1b, indent=2, default=_json_default))
    v = out1b['verdict']
    print(f'[M5] passed={v["passed"]} breadth={v["pass_a_breadth"]["breadth"]} '
          f'monotone={v["pass_a_breadth"]["monotone"]} motif={v["s1_motif"].get("op_purity")}')

    print('--- M6 control (logic_mc) ---')
    cfg_logic = MCCorpusConfig(corpus='logic_mc', cache_root=str(PHASE15_CACHE),
                               max_train_samples=SAMPLE_CAP_15, max_val_samples=2000, max_test_samples=2000)
    ctrl = run_1b_experiment(corpus_cfg=cfg_logic, train_cfg=TrainConfig(epochs=15, lr=1e-3, seed=0),
                             chain_steps=3, k_routed=128, batch_size=BATCH_PHASE15, device=DEV, seed=0)
    (mus / 'M6_control_logic.json').write_text(json.dumps(ctrl, indent=2, default=_json_default))
    print(f'[M6] control breadth-monotone={ctrl["verdict"]["pass_a_breadth"]["monotone"]} (expect False)')

if RUN_1B:
    run_section('S7 1b-musique', _musique_1b)

## Section 8 — Causal operation-specialization battery (1a flat: SWAP + LESION → col_spec)
Mirrors `04_intervention.ipynb` and saves the swap/lesion matrices + `col_spec` to
`out/phase1_5/musique/intervention_battery.json` (the notebook only printed them).

In [ ]:
def _intervention():
    import numpy as np, torch
    from experiments.phase1_5.data import (MCCorpusConfig, build_mc_corpus, encode_or_load_mc,
                               make_mc_loaders, MODE_Q_ONLY, infer_reasoning_type)
    from experiments.phase1_5.model import Phase15MoE, MOD_KG_HYPERNET
    from experiments.phase1_5.train import TrainConfig, train_phase15
    from experiments.phase1_5.eval import build_operation_labels
    from experiments.phase1_5.intervention import operation_signature, lesion_specificity, operation_swap
    from experiments.phase1_5.ablations import _json_default

    # logic_mc (LogiQA/ReClor): the regex LSAT operation axis lives here. The default
    # corpus is MuSiQue-multihop, where build_operation_labels finds no LSAT stems ([]).
    cfg = MCCorpusConfig(corpus='logic_mc', cache_root=str(PHASE15_CACHE), max_train_samples=SAMPLE_CAP_15,
                         max_val_samples=2000, max_test_samples=2000)
    corpus = build_mc_corpus(cfg)
    data = encode_or_load_mc(corpus, cfg, encoding_mode=MODE_Q_ONLY, device=DEV)
    tr, va, te = make_mc_loaders(data, batch_size=BATCH_PHASE15)
    model = Phase15MoE(d_emb=data['q_tokens'].shape[-1], d_z=256, k_routed=64,
                       modulation=MOD_KG_HYPERNET, lb_strategy='aux_free',
                       lb_target_active=4.0, routing='topk')
    res = train_phase15(model, tr, val_loader=va,
                        cfg=TrainConfig(epochs=40, lr=1e-3, k_target=4.0, lam_z=1e-3,
                                        use_best_val=False, seed=0), device=DEV)
    model = res['model']

    # build_operation_labels expects an array of operation LABELS, not question text.
    # The LSAT regex axis (infer_reasoning_type) turns each question into a label first —
    # this is exactly what regate does internally. (04_intervention passed raw questions,
    # a latent bug that was never caught because that notebook was never executed.)
    split_arr = np.asarray(data['split'])
    probe_split = 'test' if (split_arr == 'test').any() else 'val'
    pmask = split_arr == probe_split
    questions = corpus[corpus['split'] == probe_split]['question'].tolist()
    op_raw = np.array([infer_reasoning_type(q) for q in questions], dtype=object)
    labels, keep = build_operation_labels(op_raw, min_count=20)
    sub = lambda a: a[pmask][keep]
    t = lambda x: torch.from_numpy(np.asarray(x)).float()
    batch = {'q_tokens': t(sub(data['q_tokens'])), 'q_mask': t(sub(data['q_mask'])),
             'p_tokens': t(sub(data['p_tokens'])), 'p_mask': t(sub(data['p_mask'])),
             'cand_pooled': t(sub(data['cand_pooled'])),
             'answer_idx': torch.from_numpy(sub(data['answer_idx']).astype('int64'))}
    op_labels = labels
    ops = sorted(set(op_labels.tolist()))

    sigs, tops = operation_signature(model, batch['q_tokens'], batch['q_mask'], op_labels, k_top=4, device=DEV)
    swap = operation_swap(model, batch, op_labels, sigs, device=DEV)
    les = lesion_specificity(model, batch, op_labels, tops, device=DEV)
    drop = les['drop']
    col_spec = float(np.mean([drop[y][y] - np.mean([drop[x][y] for x in ops if x != y]) for y in ops]))
    diag = float(np.mean([swap[x][x] for x in ops]))
    offd = float(np.mean([swap[x][y] for x in ops for y in ops if x != y]))
    payload = {'ops': ops, 'col_spec': col_spec, 'swap_diag': diag, 'swap_offdiag': offd,
               'chance': 1.0 / len(ops), 'baseline': les['baseline'], 'swap': swap, 'drop': drop,
               'val_acc': (res['history'] or [{}])[-1].get('val_mc_acc')}
    (P15 / 'musique').mkdir(parents=True, exist_ok=True)
    (P15 / 'musique' / 'intervention_battery.json').write_text(json.dumps(payload, indent=2, default=_json_default))
    print(f'col_spec={col_spec:+.3f} | swap diag={diag:.3f} offd={offd:.3f} | chance={1.0/len(ops):.2f}')

if RUN_INTERVENTION:
    run_section('S8 intervention', _intervention)

## Section 9 — Confound-fix cells

In [ ]:
def _fix_simbench_majority():
    import numpy as np
    from collections import Counter
    from experiments.phase1.eval_simbench_classifier import load_simbench, build_targets, split_dataset
    df = load_simbench()
    dist, mask, correct = build_targets(df)
    _, _, test_idx = split_dataset(len(df), seed=42)
    n_opt = mask.sum(1)
    rand_acc = float(np.mean(1.0 / n_opt[test_idx]))
    maj_idx = Counter(correct[test_idx].tolist()).most_common(1)[0][0]
    maj_acc = float(np.mean((correct[test_idx] == maj_idx) & mask[test_idx, maj_idx]))
    out = {'random_acc': rand_acc, 'majority_acc': maj_acc,
           'mean_n_options': float(n_opt[test_idx].mean()), 'test_n': int(len(test_idx))}
    (PH1 / 'simbench_baselines.json').write_text(json.dumps(out, indent=2))
    print(f'SimBench baselines: random={rand_acc:.3f} majority={maj_acc:.3f} '
          f'mean_options={out["mean_n_options"]:.2f} (so "random 0.25" only holds for 4-option rows)')

def _fix_1a_same_axis():
    import glob
    rows = []
    paths = (glob.glob(str(P15 / 'ablations' / 'row_*.json')) +
             glob.glob(str(P15 / 'replication' / 'row_*.json')) +
             glob.glob(str(P15 / 'topk' / 'row_*.json')) +
             [str(P15 / 'musique' / 'M3_result.json')])
    for p in paths:
        try:
            r = json.loads(Path(p).read_text())
        except Exception:
            continue
        og = r.get('operation_gate') or {}
        oc = r.get('operation_ceiling_raw') or {}
        rows.append({'file': Path(p).name, 'op_gate_adj': og.get('adj_operation'),
                     'ceiling_raw_adj': oc.get('adj_operation'),
                     'n_examples': og.get('n_operation_examples'), 'gate_verdict': og.get('verdict'),
                     'ceiling_key_present': 'operation_ceiling_raw' in r})
    (P15 / 'same_axis_reconciliation.json').write_text(json.dumps(rows, indent=2, default=str))
    print(f'{"file":40s} {"gate_adj":>9} {"ceil_adj":>9} {"n":>5}')
    for r in rows:
        print(f'{r["file"][:40]:40s} {str(r["op_gate_adj"])[:9]:>9} '
              f'{str(r["ceiling_raw_adj"])[:9]:>9} {str(r["n_examples"]):>5}')

def _fix_1b_chain4():
    from experiments.phase1_5.data import MCCorpusConfig
    from experiments.phase1_5.run_1b import run_1b_experiment
    from experiments.phase1_5.ablations import _json_default
    from experiments.phase1_5.train import TrainConfig
    cfg = MCCorpusConfig(corpus='musique', cache_root=str(PHASE15_CACHE),
                         max_train_samples=SAMPLE_CAP_15, max_val_samples=2000, max_test_samples=2000)
    (P15 / 'replication').mkdir(parents=True, exist_ok=True)
    # chain_steps=4 accumulates one more step of experts + backward graph than chain=3,
    # so it OOMs at the full batch — quarter it (A100:128->32, L4:64->16).
    chain4_bs = max(8, BATCH_PHASE15 // 4)
    print(f'[chain4] batch_size={chain4_bs} (reduced from {BATCH_PHASE15} for L=4 memory)')
    for seed in SEEDS:
        out = run_1b_experiment(corpus_cfg=cfg, train_cfg=TrainConfig(epochs=15, lr=1e-3, seed=seed),
                                chain_steps=4, k_routed=128, k_active_target=4.0,
                                batch_size=chain4_bs, device=DEV, seed=seed)
        (P15 / 'replication' / f'1b_chain4_seed{seed}.json').write_text(json.dumps(out, indent=2, default=_json_default))
        v = out['verdict']
        print(f'[1b chain4 seed{seed}] breadth={v["pass_a_breadth"]["breadth"]} '
              f'monotone={v["pass_a_breadth"]["monotone"]} motif_purity={v["s1_motif"].get("op_purity")}')

def _fix_1b_motif():
    import glob
    rows = []
    for p in glob.glob(str(P15 / 'musique' / 'M5_*.json')) + glob.glob(str(P15 / 'replication' / '1b_chain4_*.json')):
        try:
            v = json.loads(Path(p).read_text())['verdict']
        except Exception:
            continue
        s1 = v.get('s1_motif', {})
        rows.append({'file': Path(p).name, 'op_purity': s1.get('op_purity'),
                     'op_chance': s1.get('op_chance'), 'op_above_chance': s1.get('op_above_chance')})
        print(f'{Path(p).name:30s} purity={s1.get("op_purity")} chance={s1.get("op_chance")} '
              f'above={s1.get("op_above_chance")}')
    print('NOTE: evaluate_1b measures motif vs chance only (raw text for a topic control is not in '
          'the encoded data); op_beats_topic is NOT asserted here.')
    return rows

def _confound_fixes():
    if FIX_SIMBENCH_MAJORITY:
        print('\n# 9.1 SimBench majority/base-rate'); _fix_simbench_majority()
    if FIX_1A_SAME_AXIS:
        print('\n# 9.2 1a same-axis ceiling reconciliation'); _fix_1a_same_axis()
    if FIX_1B_CHAINSTEPS_4:
        print('\n# 9.3 1b chain_steps=4 + multi-seed'); _fix_1b_chain4()
    if FIX_1B_MOTIF_TOPIC_CTRL:
        print('\n# 9.4 1b motif purity vs chance'); _fix_1b_motif()

if INCLUDE_CONFOUND_FIXES:
    run_section('S9 confound-fixes', _confound_fixes)

## Section 10 — VERIFICATION (documented vs measured)
Loads every saved JSON and prints a consolidated table, then writes `out/VERIFICATION.json`.
Documented values are the audit-prose figures; for Phase-1 recon/SimBench a non-trivial delta
is EXPECTED when `PHASE1_RECON_CORPUS='default4'` (the recorded numbers are pre-flip 7-source).

In [ ]:
def _verification():
    rows = []

    def add(experiment, metric, documented, measured, path, note=''):
        delta = None
        try:
            if measured is not None and documented is not None:
                delta = round(float(measured) - float(documented), 4)
        except (TypeError, ValueError):
            delta = None
        rows.append({'experiment': experiment, 'metric': metric, 'documented': documented,
                     'measured': measured, 'delta': delta, 'path': path, 'note': note})

    def load(p):
        p = Path(p)
        return json.loads(p.read_text()) if p.exists() else None

    DOC = {'ph1_v3_minimal': 0.8864, 'ph1_v4_diverse': 0.8786, 'ph1_v5_arch': 0.8788,
           'ph1_v6_long': 0.8789, 'b0_v0': 0.8687}
    DOC_SB = {'ph1_v3_minimal': 0.7120, 'ph1_v4_diverse': 0.7029, 'ph1_v5_arch': 0.7067,
              'ph1_v6_long': 0.6901, 'b0_v0': 0.7082, 'b1_v0': None}
    cnote = 'corpus=default4; documented is pre-flip 7-source' if PHASE1_RECON_CORPUS == 'default4' else ''

    for base, doc in DOC.items():
        ev = load(PH1 / rid(base) / 'eval.json')
        meas = ev['cycle_reconstruction']['mean_cosine'] if ev and 'cycle_reconstruction' in ev else None
        add(base, 'recon_cosine', doc, meas, str(PH1 / rid(base) / 'eval.json'), cnote)
    for base, doc in DOC_SB.items():
        sb = load(PH1 / rid(base) / 'simbench_eval.json')
        meas = sb['test']['argmax_acc'] if sb else None
        add(base, 'simbench_argmax_acc', doc, meas, str(PH1 / rid(base) / 'simbench_eval.json'), cnote)

    b1h = load(PH1 / rid('b1_v0') / 'history.json')
    b1s = load(PH1 / rid('b1_v0') / 'sanity.json')
    if b1h:
        add('b1_v0', 'K_active_last(of 20)', 'predicted collapse(~1)',
            round(b1h[-1].get('active_frac', 0) * 20, 2), str(PH1 / rid('b1_v0') / 'history.json'),
            'trained through; measured')
    elif b1s:
        add('b1_v0', 'sanity verdict', 'predicted collapse',
            f"{b1s.get('verdict')} k_act={b1s.get('k_active_mean')}", str(PH1 / rid('b1_v0') / 'sanity.json'),
            'sanity-FAILED = collapse confirmed (measured)')
    else:
        add('b1_v0', 'K_active', 'predicted collapse(~1)', None, '(none)', 'did not run')

    sbb = load(PH1 / 'simbench_baselines.json')
    if sbb:
        add('simbench', 'random_baseline', None, round(sbb['random_acc'], 4), 'out/phase1/simbench_baselines.json',
            f'mean_options={sbb["mean_n_options"]:.2f}')
        add('simbench', 'majority_baseline', None, round(sbb['majority_acc'], 4), 'out/phase1/simbench_baselines.json')

    rep = load(PH1 / 'engine_a' / 'report_stage1.json')
    if rep:
        add('engine_a', 'adj_operation', 0.176, rep.get('adj_operation'), 'out/phase1/engine_a/report_stage1.json',
            f'verdict={rep.get("verdict")}')
        add('engine_a', 'ceiling_0b_adj', 0.60, rep.get('ceiling_0b_adj'), 'out/phase1/engine_a/report_stage1.json')

    import glob
    for p in sorted(glob.glob(str(P15 / 'ablations' / 'row_A.*_seed0_*.json'))):
        r = load(p); og = (r or {}).get('operation_gate') or {}; oc = (r or {}).get('operation_ceiling_raw') or {}
        add(f'1a {r.get("row_id")}', 'op_gate_adj vs ceiling', f'ceil~{oc.get("adj_operation")}',
            og.get('adj_operation'), p, f'sigma_gate={(r or {}).get("gate", {}).get("passes_sigma_gate")}')

    m5 = load(P15 / 'musique' / 'M5_chain3_seed0.json')
    if m5:
        v = m5['verdict']
        add('1b M5 chain3', 'pass_a_breadth', '{2,2,2} non-monotone', str(v['pass_a_breadth']['breadth']),
            'out/phase1_5/musique/M5_chain3_seed0.json', f'monotone={v["pass_a_breadth"]["monotone"]}')
    for seed in SEEDS:
        c4 = load(P15 / 'replication' / f'1b_chain4_seed{seed}.json')
        if c4:
            v = c4['verdict']
            add(f'1b chain4 s{seed}', 'pass_a_breadth', 'cap removed (L=4)', str(v['pass_a_breadth']['breadth']),
                f'out/phase1_5/replication/1b_chain4_seed{seed}.json', f'monotone={v["pass_a_breadth"]["monotone"]}')

    ib = load(P15 / 'musique' / 'intervention_battery.json')
    if ib:
        add('intervention', 'col_spec', '~0.05 (weak)', round(ib['col_spec'], 4),
            'out/phase1_5/musique/intervention_battery.json', f'chance={ib["chance"]:.2f}')

    print(f'{"experiment":18s} {"metric":24s} {"documented":>22} {"measured":>14} {"delta":>9}  note')
    print('-' * 120)
    for r in rows:
        print(f'{str(r["experiment"])[:18]:18s} {str(r["metric"])[:24]:24s} '
              f'{str(r["documented"])[:22]:>22} {str(r["measured"])[:14]:>14} '
              f'{str(r["delta"]):>9}  {r["note"]}')

    Path('out').mkdir(exist_ok=True)
    Path('out/VERIFICATION.json').write_text(json.dumps(
        {'section_status': RESULTS, 'corpus': PHASE1_RECON_CORPUS, 'tier': TIER, 'rows': rows},
        indent=2, default=str))
    print('\nsaved -> out/VERIFICATION.json')
    print('\nsection status:', RESULTS)

run_section('S10 verification', _verification)